In [2]:
from pathlib import Path
import pandas as pd

PREPARED_DIR = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\prepared"
)

def load_analysis_sample(source):
    frame = pd.read_csv(
        PREPARED_DIR / f"time_tax_{source}_matched.csv.gz",
        compression="gzip",
        keep_default_na=False,
        na_values=[""],
        dtype={
            "iso3": "string",
            "country": "string",
            "source": "string",
            "year": "Int64",
            "cross_source_conflict": "boolean",
        },
    )

    if frame.duplicated(["iso3", "year"]).any():
        raise ValueError(f"Duplicate country-year keys: {source}")

    if frame["cross_source_conflict"].isna().any():
        raise ValueError(f"Missing conflict flags: {source}")

    return frame


sdg_data = load_analysis_sample("sdg")
gender_data = load_analysis_sample("gender")

for name, frame in [("SDG", sdg_data), ("Gender", gender_data)]:
    print(
        f"{name}: {len(frame)} observations, "
        f"{frame['iso3'].nunique()} entities, "
        f"{frame['year'].min()}–{frame['year'].max()}"
    )

SDG: 177 observations, 89 entities, 2000–2019
Gender: 66 observations, 27 entities, 2000–2024


In [3]:
import numpy as np
import pandas as pd

eda = sdg_data.copy()

COLUMN_LABELS = {
    "gender__SL.TLF.CACT.FE.ZS": "female_lfpr_pct",
    "gender__SL.TLF.CACT.MA.ZS": "male_lfpr_pct",
    "wdi__EG.ELC.ACCS.ZS": "electricity_pct",
    "wdi__SH.H2O.BASW.ZS": "water_pct",
    "wdi__EG.CFT.ACCS.ZS": "clean_cooking_pct",
    "wdi__NY.GDP.PCAP.PP.KD": "gdp_per_capita_ppp",
    "wdi__SP.DYN.TFRT.IN": "fertility_rate",
    "wdi__SP.URB.TOTL.IN.ZS": "urban_population_pct",
}

eda = eda.rename(columns=COLUMN_LABELS)

eda["lfpr_gap_pp"] = (
    eda["male_lfpr_pct"] - eda["female_lfpr_pct"]
)

expected_time_gap = (
    eda["female_unpaid_hours"] - eda["male_unpaid_hours"]
)

if not np.allclose(
    eda["time_gap_hours"].to_numpy(dtype=float),
    expected_time_gap.to_numpy(dtype=float),
    rtol=0,
    atol=1e-8,
    equal_nan=True,
):
    raise ValueError("Stored time gaps do not match female minus male hours.")

RANGE_RULES = {
    "female_unpaid_hours": (0, 24),
    "male_unpaid_hours": (0, 24),
    "time_gap_hours": (-24, 24),
    "female_lfpr_pct": (0, 100),
    "male_lfpr_pct": (0, 100),
    "lfpr_gap_pp": (-100, 100),
    "electricity_pct": (0, 100),
    "water_pct": (0, 100),
    "clean_cooking_pct": (0, 100),
    "urban_population_pct": (0, 100),
    "fertility_rate": (0, np.inf),
    "gdp_per_capita_ppp": (0, np.inf),
}

quality_rows = []
flagged_parts = []

for variable, (lower, upper) in RANGE_RULES.items():
    values = pd.to_numeric(eda[variable], errors="raise")
    eda[variable] = values

    finite = pd.Series(
        np.isfinite(values.to_numpy(dtype=float, na_value=np.nan)),
        index=eda.index,
    )

    flagged = values.notna() & (
        ~finite | ~values.between(lower, upper).fillna(False)
    )

    if variable == "gdp_per_capita_ppp":
        flagged = flagged | values.eq(0).fillna(False)

    quality_rows.append({
        "variable": variable,
        "nonmissing": int(values.notna().sum()),
        "missing": int(values.isna().sum()),
        "minimum": values.min(),
        "maximum": values.max(),
        "flagged": int(flagged.sum()),
    })

    if flagged.any():
        details = eda.loc[
            flagged, ["iso3", "country", "year", variable]
        ].rename(columns={variable: "value"})
        details["variable"] = variable
        flagged_parts.append(details)

quality_report = pd.DataFrame(quality_rows)
display(quality_report.round(3))

if flagged_parts:
    flagged_records = pd.concat(flagged_parts, ignore_index=True)
    display(flagged_records)
else:
    print("No violations found under the checked range rules.")

,variable,nonmissing,missing,minimum,maximum,flagged
0,female_unpaid_hours,177,0,1.205,7.450,0
1,male_unpaid_hours,177,0,0.300,4.167,0
2,time_gap_hours,177,0,0.502,5.462,0
3,female_lfpr_pct,177,0,12.290,87.119,0
4,male_lfpr_pct,177,0,56.075,95.596,0
5,lfpr_gap_pp,177,0,-1.070,60.445,0
6,electricity_pct,177,0,5.300,100.000,0
7,water_pct,174,3,36.920,100.000,0
8,clean_cooking_pct,171,6,0.400,100.000,0
9,urban_population_pct,177,0,15.104,100.000,0


No violations found under the checked range rules.


In [4]:
DESCRIPTIVE_VARIABLES = [
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "female_lfpr_pct",
    "male_lfpr_pct",
    "lfpr_gap_pp",
    "electricity_pct",
    "water_pct",
    "clean_cooking_pct",
    "gdp_per_capita_ppp",
    "fertility_rate",
    "urban_population_pct",
]

latest_per_entity = (
    eda.sort_values(["iso3", "year"])
    .drop_duplicates("iso3", keep="last")
    .copy()
)

eda_views = {
    "All observations": eda,
    "Exclude source conflicts": eda.loc[
        ~eda["cross_source_conflict"]
    ].copy(),
    "Latest observation per entity": latest_per_entity,
}

descriptive_tables = {}
sample_rows = []

for sample_name, frame in eda_views.items():
    summary = (
        frame[DESCRIPTIVE_VARIABLES]
        .describe(percentiles=[0.25, 0.50, 0.75])
        .T
        .rename(columns={"count": "nonmissing", "50%": "median"})
    )
    summary["missing"] = frame[DESCRIPTIVE_VARIABLES].isna().sum()
    descriptive_tables[sample_name] = summary

    sample_rows.append({
        "sample": sample_name,
        "observations": len(frame),
        "entities": frame["iso3"].nunique(),
        "first_year": frame["year"].min(),
        "last_year": frame["year"].max(),
        "flagged_pairs": int(frame["cross_source_conflict"].sum()),
    })

display(pd.DataFrame(sample_rows))

for sample_name, summary in descriptive_tables.items():
    print(f"\n{sample_name}")
    display(summary.round(3))

print("\nTime-gap direction in all observations")
gap = eda["time_gap_hours"]

display(pd.DataFrame({
    "direction": ["Female higher", "Approximately equal", "Male higher"],
    "observations": [
        int(gap.gt(1e-8).sum()),
        int(gap.abs().le(1e-8).sum()),
        int(gap.lt(-1e-8).sum()),
    ],
}))

,sample,observations,entities,first_year,last_year,flagged_pairs
0,All observations,177,89,2000,2019,5
1,Exclude source conflicts,172,89,2000,2019,0
2,Latest observation per entity,89,89,2001,2019,3



All observations


,nonmissing,mean,std,min,25%,median,75%,max,missing
female_unpaid_hours,177.0,4.285,0.983,1.205,3.700,4.151,4.814,7.450,0
male_unpaid_hours,177.0,1.812,0.754,0.300,1.133,1.955,2.350,4.167,0
time_gap_hours,177.0,2.474,0.991,0.502,1.640,2.383,3.050,5.462,0
female_lfpr_pct,177.0,51.633,12.421,12.290,45.905,53.834,58.224,87.119,0
male_lfpr_pct,177.0,71.653,6.817,56.075,66.967,71.814,76.035,95.596,0
lfpr_gap_pp,177.0,20.020,12.258,-1.070,11.953,15.630,24.100,60.445,0
electricity_pct,177.0,92.984,18.671,5.300,98.200,99.800,100.000,100.000,0
water_pct,174.0,92.656,11.863,36.920,90.965,97.779,99.631,100.000,3
clean_cooking_pct,171.0,82.866,28.702,0.400,77.100,100.000,100.000,100.000,6
gdp_per_capita_ppp,175.0,32355.265,24621.510,1214.743,12809.623,23515.347,51741.398,138046.596,2



Exclude source conflicts


,nonmissing,mean,std,min,25%,median,75%,max,missing
female_unpaid_hours,172.0,4.321,0.958,1.510,3.727,4.222,4.833,7.450,0
male_unpaid_hours,172.0,1.817,0.756,0.300,1.125,1.969,2.353,4.167,0
time_gap_hours,172.0,2.504,0.984,0.714,1.695,2.400,3.088,5.462,0
female_lfpr_pct,172.0,51.418,12.517,12.290,45.658,53.620,58.166,87.119,0
male_lfpr_pct,172.0,71.555,6.871,56.075,66.916,71.570,75.943,95.596,0
lfpr_gap_pp,172.0,20.137,12.381,-1.070,11.957,15.632,24.202,60.445,0
electricity_pct,172.0,92.797,18.909,5.300,98.050,99.850,100.000,100.000,0
water_pct,169.0,92.563,11.999,36.920,90.955,97.653,99.668,100.000,3
clean_cooking_pct,166.0,82.718,28.958,0.400,76.900,100.000,100.000,100.000,6
gdp_per_capita_ppp,170.0,32416.203,24751.188,1214.743,12746.502,24257.409,51018.125,138046.596,2



Latest observation per entity


,nonmissing,mean,std,min,25%,median,75%,max,missing
female_unpaid_hours,89.0,4.229,0.940,1.205,3.683,4.383,4.833,6.664,0
male_unpaid_hours,89.0,1.704,0.735,0.300,1.049,1.800,2.283,3.079,0
time_gap_hours,89.0,2.525,1.008,0.502,1.800,2.400,3.102,4.783,0
female_lfpr_pct,89.0,50.622,14.252,12.290,44.760,53.302,58.668,80.089,0
male_lfpr_pct,89.0,72.117,7.559,58.057,66.967,71.384,77.095,95.596,0
lfpr_gap_pp,89.0,21.495,14.188,-1.070,11.641,15.630,27.602,60.445,0
electricity_pct,89.0,90.647,22.923,5.300,97.200,99.600,100.000,100.000,0
water_pct,89.0,91.811,12.592,38.919,89.574,96.743,99.668,100.000,0
clean_cooking_pct,86.0,80.984,30.861,0.400,76.600,97.700,100.000,100.000,3
gdp_per_capita_ppp,88.0,28946.291,25673.366,1214.743,11895.810,19819.859,43253.301,138046.596,1



Time-gap direction in all observations


,direction,observations
0,Female higher,177
1,Approximately equal,0
2,Male higher,0


In [5]:
time_detail_columns = [
    "iso3",
    "country",
    "year",
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "female_lfpr_pct",
    "lfpr_gap_pp",
    "cross_source_conflict",
]

print("Largest time gaps: country-year observations")
display(
    eda.nlargest(10, "time_gap_hours")[time_detail_columns].round(3)
)

print("Smallest time gaps: country-year observations")
display(
    eda.nsmallest(10, "time_gap_hours")[time_detail_columns].round(3)
)

for variable in ["electricity_pct", "water_pct", "clean_cooking_pct"]:
    print(f"\nLowest observed values: {variable}")

    display(
        eda.dropna(subset=[variable])
        .nsmallest(5, variable)[
            [
                "iso3",
                "country",
                "year",
                variable,
                "time_gap_hours",
                "cross_source_conflict",
            ]
        ]
        .round(3)
    )

print("\nYear distribution: latest observation per entity")
display(
    latest_per_entity.groupby("year")
    .size()
    .rename("entities")
    .to_frame()
)

Largest time gaps: country-year observations


,iso3,country,year,female_unpaid_hours,male_unpaid_hours,time_gap_hours,female_lfpr_pct,lfpr_gap_pp,cross_source_conflict
103,MEX,Mexico,2002,6.987,1.525,5.462,38.314,42.067,False
104,MEX,Mexico,2009,7.450,2.492,4.958,43.080,36.386,False
135,PSE,West Bank and Gaza,2000,5.550,0.767,4.783,12.402,55.565,False
47,EGY,"Egypt, Arab Rep.",2015,5.367,0.583,4.783,22.813,48.157,False
70,IRQ,Iraq,2007,5.783,1.000,4.783,12.290,60.445,False
105,MEX,Mexico,2014,7.085,2.344,4.742,43.550,35.246,False
4,ARM,Armenia,2004,5.767,1.100,4.667,53.518,16.511,False
150,TUN,Tunisia,2006,5.267,0.650,4.617,24.489,44.277,False
151,TUR,Turkey,2006,5.281,0.852,4.428,23.426,45.964,False
0,ALB,Albania,2011,5.217,0.833,4.383,52.561,15.068,False


Smallest time gaps: country-year observations


,iso3,country,year,female_unpaid_hours,male_unpaid_hours,time_gap_hours,female_lfpr_pct,lfpr_gap_pp,cross_source_conflict
36,COL,Colombia,2017,1.205,0.703,0.502,54.343,23.852,True
124,NOR,Norway,2011,3.674,2.960,0.714,63.764,6.783,False
147,SWE,Sweden,2011,3.850,3.079,0.771,59.526,8.087,False
95,LAO,Lao PDR,2017,3.265,2.415,0.850,56.020,7.522,False
96,LBR,Liberia,2010,1.510,0.618,0.892,71.772,10.869,False
43,DNK,Denmark,2009,3.750,2.700,1.050,59.768,9.632,False
42,DNK,Denmark,2001,3.500,2.433,1.067,59.874,11.318,False
54,FIN,Finland,2010,3.649,2.518,1.131,55.871,7.896,False
146,SWE,Sweden,2001,4.236,3.047,1.189,58.578,8.378,False
23,CAN,Canada,2016,3.500,2.300,1.200,60.879,9.647,True



Lowest observed values: electricity_pct


...

,iso3,country,year,water_pct,time_gap_hours,cross_source_conflict
153,TZA,Tanzania,2006,36.920,3.000,False
52,ETH,Ethiopia,2013,38.919,3.050,False
154,TZA,Tanzania,2014,49.128,2.933,False
155,UGA,Uganda,2018,52.965,1.700,False
92,LAO,Lao PDR,2003,54.865,1.800,False



Lowest observed values: clean_cooking_pct


,iso3,country,year,clean_cooking_pct,time_gap_hours,cross_source_conflict
96,LBR,Liberia,2010,0.4,0.892,False
153,TZA,Tanzania,2006,0.9,3.000,False
155,UGA,Uganda,2018,1.2,1.700,False
92,LAO,Lao PDR,2003,1.7,1.800,False
117,MWI,Malawi,2005,2.2,1.783,False



Year distribution: latest observation per entity


,entities
year,
2001,2
2002,1
2003,5
2004,1
2005,2
2006,2
2007,2
2008,3
2009,5


In [6]:
TIME_VARIABLES = [
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
]

RELATED_VARIABLES = [
    "female_lfpr_pct",
    "lfpr_gap_pp",
    "electricity_pct",
    "water_pct",
    "clean_cooking_pct",
]

association_views = {
    "All available observations": eda.copy(),
    "Exclude source conflicts": eda.loc[
        ~eda["cross_source_conflict"]
    ].copy(),
    "Latest observation per entity": latest_per_entity.copy(),
}

association_rows = []

for view_name, frame in association_views.items():
    for time_variable in TIME_VARIABLES:
        for related_variable in RELATED_VARIABLES:
            pair = (
                frame[
                    ["iso3", "year", time_variable, related_variable]
                ]
                .replace([np.inf, -np.inf], np.nan)
                .dropna(subset=[time_variable, related_variable])
            )

            x = pair[time_variable]
            y = pair[related_variable]

            usable = (
                len(pair) >= 3
                and x.nunique() > 1
                and y.nunique() > 1
            )

            # Pearson measures linear association.
            pearson = x.corr(y) if usable else np.nan

            # Pearson correlation of average ranks equals Spearman rho.
            # This avoids requiring an additional scipy installation.
            spearman = (
                x.rank(method="average").corr(y.rank(method="average"))
                if usable else np.nan
            )

            association_rows.append({
                "view": view_name,
                "time_variable": time_variable,
                "related_variable": related_variable,
                "observations": len(pair),
                "entities": pair["iso3"].nunique(),
                "pearson_r": pearson,
                "spearman_rho": spearman,
            })

associations = pd.DataFrame(association_rows)

for view_name in association_views:
    print(f"\n{view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            associations.loc[
                associations["view"].eq(view_name)
            ].drop(columns="view").reset_index(drop=True).round(3)
        )


All available observations


,time_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,female_unpaid_hours,female_lfpr_pct,177,89,-0.354,-0.397
1,female_unpaid_hours,lfpr_gap_pp,177,89,0.337,0.360
2,female_unpaid_hours,electricity_pct,177,89,0.264,-0.158
3,female_unpaid_hours,water_pct,174,89,0.074,-0.198
4,female_unpaid_hours,clean_cooking_pct,171,86,0.183,-0.090
5,male_unpaid_hours,female_lfpr_pct,177,89,0.271,0.293
6,male_unpaid_hours,lfpr_gap_pp,177,89,-0.441,-0.418
7,male_unpaid_hours,electricity_pct,177,89,0.322,0.414
8,male_unpaid_hours,water_pct,174,89,0.293,0.413
9,male_unpaid_hours,clean_cooking_pct,171,86,0.380,0.446



Exclude source conflicts


,time_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,female_unpaid_hours,female_lfpr_pct,172,89,-0.357,-0.397
1,female_unpaid_hours,lfpr_gap_pp,172,89,0.351,0.371
2,female_unpaid_hours,electricity_pct,172,89,0.287,-0.165
3,female_unpaid_hours,water_pct,169,89,0.089,-0.208
4,female_unpaid_hours,clean_cooking_pct,166,86,0.203,-0.099
5,male_unpaid_hours,female_lfpr_pct,172,89,0.275,0.300
6,male_unpaid_hours,lfpr_gap_pp,172,89,-0.440,-0.411
7,male_unpaid_hours,electricity_pct,172,89,0.328,0.404
8,male_unpaid_hours,water_pct,169,89,0.295,0.408
9,male_unpaid_hours,clean_cooking_pct,166,86,0.384,0.438



Latest observation per entity


,time_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,female_unpaid_hours,female_lfpr_pct,89,89,-0.428,-0.431
1,female_unpaid_hours,lfpr_gap_pp,89,89,0.333,0.437
2,female_unpaid_hours,electricity_pct,89,89,0.327,-0.078
3,female_unpaid_hours,water_pct,89,89,0.132,-0.147
4,female_unpaid_hours,clean_cooking_pct,86,86,0.298,0.034
5,male_unpaid_hours,female_lfpr_pct,89,89,0.241,0.226
6,male_unpaid_hours,lfpr_gap_pp,89,89,-0.482,-0.428
7,male_unpaid_hours,electricity_pct,89,89,0.331,0.402
8,male_unpaid_hours,water_pct,89,89,0.319,0.420
9,male_unpaid_hours,clean_cooking_pct,86,86,0.395,0.487


In [7]:
time_tax_results = associations.loc[
    associations["time_variable"].eq("time_gap_hours")
].copy()

for metric in ["pearson_r", "spearman_rho"]:
    comparison = (
        time_tax_results.pivot(
            index="related_variable",
            columns="view",
            values=metric,
        )
        .reindex(columns=list(association_views))
    )

    print(f"\nTime Tax association comparison: {metric}")
    display(comparison.round(3))

print("\nObservation counts used in each comparison")
display(
    time_tax_results.pivot(
        index="related_variable",
        columns="view",
        values="observations",
    ).reindex(columns=list(association_views))
)


Time Tax association comparison: pearson_r


view,All available observations,Exclude source conflicts,Latest observation per entity
related_variable,,,
clean_cooking_pct,-0.107,-0.096,-0.010
electricity_pct,0.016,0.028,0.064
female_lfpr_pct,-0.557,-0.559,-0.574
lfpr_gap_pp,0.670,0.681,0.661
water_pct,-0.149,-0.140,-0.110



Time Tax association comparison: spearman_rho


view,All available observations,Exclude source conflicts,Latest observation per entity
related_variable,,,
clean_cooking_pct,-0.414,-0.422,-0.290
electricity_pct,-0.431,-0.437,-0.318
female_lfpr_pct,-0.527,-0.531,-0.503
lfpr_gap_pp,0.625,0.640,0.637
water_pct,-0.473,-0.481,-0.393



Observation counts used in each comparison


view,All available observations,Exclude source conflicts,Latest observation per entity
related_variable,,,
clean_cooking_pct,171,166,86
electricity_pct,177,172,89
female_lfpr_pct,177,172,89
lfpr_gap_pp,177,172,89
water_pct,174,169,89


In [8]:
INFRASTRUCTURE_VARIABLES = [
    "electricity_pct",
    "water_pct",
    "clean_cooking_pct",
]

LOO_VARIABLES = [
    "female_lfpr_pct",
    "lfpr_gap_pp",
] + INFRASTRUCTURE_VARIABLES


def calculate_correlation(frame, x_column, y_column, method):
    pair = (
        frame[[x_column, y_column]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if (
        len(pair) < 3
        or pair[x_column].nunique() < 2
        or pair[y_column].nunique() < 2
    ):
        return np.nan

    x = pair[x_column]
    y = pair[y_column]

    if method == "pearson":
        return x.corr(y)

    if method == "spearman":
        return x.rank(method="average").corr(
            y.rank(method="average")
        )

    raise ValueError(f"Unsupported method: {method}")


loo_rows = []
loo_summary_rows = []

for view_name, frame in association_views.items():
    for related_variable in LOO_VARIABLES:
        usable = (
            frame[
                ["iso3", "country", "time_gap_hours", related_variable]
            ]
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["time_gap_hours", related_variable])
        )

        for method in ["pearson", "spearman"]:
            baseline = calculate_correlation(
                usable, "time_gap_hours", related_variable, method
            )

            current_rows = []

            for iso3 in sorted(usable["iso3"].unique()):
                remaining = usable.loc[usable["iso3"].ne(iso3)]

                coefficient = calculate_correlation(
                    remaining, "time_gap_hours", related_variable, method
                )

                country = usable.loc[
                    usable["iso3"].eq(iso3), "country"
                ].iloc[0]

                result = {
                    "view": view_name,
                    "related_variable": related_variable,
                    "method": method,
                    "excluded_iso3": iso3,
                    "excluded_country": country,
                    "removed_observations": len(usable) - len(remaining),
                    "baseline": baseline,
                    "coefficient_after_exclusion": coefficient,
                    "absolute_change": abs(coefficient - baseline),
                }

                current_rows.append(result)
                loo_rows.append(result)

            results = pd.DataFrame(current_rows)
            valid_results = results.dropna(
                subset=["coefficient_after_exclusion"]
            )

            if valid_results.empty:
                continue

            largest_change = valid_results.loc[
                valid_results["absolute_change"].idxmax()
            ]

            loo_summary_rows.append({
                "view": view_name,
                "related_variable": related_variable,
                "method": method,
                "baseline": baseline,
                "minimum_after_exclusion":
                    valid_results["coefficient_after_exclusion"].min(),
                "maximum_after_exclusion":
                    valid_results["coefficient_after_exclusion"].max(),
                "largest_absolute_change":
                    largest_change["absolute_change"],
                "most_influential_exclusion":
                    largest_change["excluded_country"],
            })

loo_details = pd.DataFrame(loo_rows)
loo_summary = pd.DataFrame(loo_summary_rows)

for view_name in association_views:
    print(f"\nLeave-one-entity-out sensitivity: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            loo_summary.loc[loo_summary["view"].eq(view_name)]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Leave-one-entity-out sensitivity: All available observations


,related_variable,method,baseline,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion
0,female_lfpr_pct,pearson,-0.557,-0.596,-0.528,0.039,Tanzania
1,female_lfpr_pct,spearman,-0.527,-0.578,-0.503,0.051,Kazakhstan
2,lfpr_gap_pp,pearson,0.670,0.650,0.692,0.022,Armenia
3,lfpr_gap_pp,spearman,0.625,0.604,0.665,0.040,Kazakhstan
4,electricity_pct,pearson,0.016,-0.028,0.061,0.045,United States
5,electricity_pct,spearman,-0.431,-0.476,-0.332,0.100,United States
6,water_pct,pearson,-0.149,-0.196,-0.100,0.048,United States
7,water_pct,spearman,-0.473,-0.522,-0.418,0.055,United States
8,clean_cooking_pct,pearson,-0.107,-0.178,-0.043,0.072,Lao PDR
9,clean_cooking_pct,spearman,-0.414,-0.469,-0.318,0.096,United States



Leave-one-entity-out sensitivity: Exclude source conflicts


,related_variable,method,baseline,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion
0,female_lfpr_pct,pearson,-0.559,-0.598,-0.529,0.039,Tanzania
1,female_lfpr_pct,spearman,-0.531,-0.582,-0.507,0.051,Kazakhstan
2,lfpr_gap_pp,pearson,0.681,0.662,0.704,0.023,Armenia
3,lfpr_gap_pp,spearman,0.640,0.619,0.681,0.041,Kazakhstan
4,electricity_pct,pearson,0.028,-0.018,0.078,0.050,United States
5,electricity_pct,spearman,-0.437,-0.485,-0.330,0.107,United States
6,water_pct,pearson,-0.140,-0.189,-0.087,0.053,United States
7,water_pct,spearman,-0.481,-0.534,-0.421,0.060,United States
8,clean_cooking_pct,pearson,-0.096,-0.171,-0.026,0.075,Lao PDR
9,clean_cooking_pct,spearman,-0.422,-0.481,-0.319,0.103,United States



Leave-one-entity-out sensitivity: Latest observation per entity


,related_variable,method,baseline,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion
0,female_lfpr_pct,pearson,-0.574,-0.602,-0.543,0.031,Iraq
1,female_lfpr_pct,spearman,-0.503,-0.535,-0.486,0.031,Peru
2,lfpr_gap_pp,pearson,0.661,0.637,0.693,0.031,Qatar
3,lfpr_gap_pp,spearman,0.637,0.624,0.670,0.033,Azerbaijan
4,electricity_pct,pearson,0.064,-0.006,0.083,0.070,Liberia
5,electricity_pct,spearman,-0.318,-0.362,-0.303,0.044,Liberia
6,water_pct,pearson,-0.110,-0.148,-0.090,0.038,Liberia
7,water_pct,spearman,-0.393,-0.433,-0.376,0.040,Liberia
8,clean_cooking_pct,pearson,-0.010,-0.063,0.026,0.054,Liberia
9,clean_cooking_pct,spearman,-0.290,-0.335,-0.275,0.045,Liberia


In [9]:
COVERAGE_BINS = [0, 50, 80, 95, np.inf]
COVERAGE_LABELS = [
    "<50%",
    "50–<80%",
    "80–<95%",
    "95–100%",
]

grouping_views = {
    "All available observations": eda,
    "Latest observation per entity": latest_per_entity,
}

infrastructure_group_tables = {}

for view_name, frame in grouping_views.items():
    for variable in INFRASTRUCTURE_VARIABLES:
        valid = frame.dropna(
            subset=[
                variable,
                "female_unpaid_hours",
                "male_unpaid_hours",
                "time_gap_hours",
            ]
        ).copy()

        if not valid[variable].between(0, 100).all():
            raise ValueError(f"Values outside 0–100: {variable}")

        valid["coverage_band"] = pd.cut(
            valid[variable],
            bins=COVERAGE_BINS,
            labels=COVERAGE_LABELS,
            right=False,
            include_lowest=True,
        )

        summary = (
            valid.groupby("coverage_band", observed=True)
            .agg(
                observations=("iso3", "size"),
                entities=("iso3", "nunique"),
                first_year=("year", "min"),
                last_year=("year", "max"),
                female_hours_mean=("female_unpaid_hours", "mean"),
                male_hours_mean=("male_unpaid_hours", "mean"),
                time_gap_mean=("time_gap_hours", "mean"),
                time_gap_median=("time_gap_hours", "median"),
                time_gap_q25=(
                    "time_gap_hours", lambda values: values.quantile(0.25)
                ),
                time_gap_q75=(
                    "time_gap_hours", lambda values: values.quantile(0.75)
                ),
                flagged_pairs=("cross_source_conflict", "sum"),
            )
            .reset_index()
        )

        infrastructure_group_tables[(view_name, variable)] = summary

        print(f"\n{view_name} | {variable}")
        with pd.option_context("display.max_columns", None):
            display(summary.round(3))

        print(
            "Observations without infrastructure values:",
            int(frame[variable].isna().sum()),
        )


All available observations | electricity_pct


,coverage_band,observations,entities,first_year,last_year,female_hours_mean,male_hours_mean,time_gap_mean,time_gap_median,time_gap_q25,time_gap_q75,flagged_pairs
0,<50%,9,8,2003,2018,3.212,0.978,2.234,2.250,1.783,2.933,0
1,50–<80%,11,8,2000,2014,3.934,1.452,2.483,2.617,2.175,2.687,0
2,80–<95%,13,11,2000,2017,4.496,1.642,2.854,2.700,2.391,3.577,0
3,95–100%,144,70,2000,2019,4.360,1.906,2.453,2.317,1.564,3.106,5


Observations without infrastructure values: 0

All available observations | water_pct


,coverage_band,observations,entities,first_year,last_year,female_hours_mean,male_hours_mean,time_gap_mean,time_gap_median,time_gap_q25,time_gap_q75,flagged_pairs
0,<50%,3,2,2006,2014,4.228,1.233,2.994,3.000,2.967,3.025,0
1,50–<80%,15,10,2000,2018,3.487,1.124,2.363,2.583,1.850,2.700,0
2,80–<95%,44,31,2000,2019,4.865,1.780,3.084,2.895,2.371,3.629,1
3,95–100%,112,55,2000,2019,4.167,1.917,2.250,2.174,1.470,2.821,4


Observations without infrastructure values: 3

All available observations | clean_cooking_pct


...

Observations without infrastructure values: 0

Latest observation per entity | water_pct


,coverage_band,observations,entities,first_year,last_year,female_hours_mean,male_hours_mean,time_gap_mean,time_gap_median,time_gap_q25,time_gap_q75,flagged_pairs
0,<50%,2,2,2013,2014,4.292,1.300,2.992,2.992,2.962,3.021,0
1,50–<80%,8,8,2003,2018,3.293,0.927,2.366,2.433,1.763,2.700,0
2,80–<95%,24,24,2001,2019,4.691,1.648,3.043,2.770,2.317,3.988,0
3,95–100%,55,55,2001,2019,4.162,1.856,2.305,2.267,1.528,2.882,3


Observations without infrastructure values: 0

Latest observation per entity | clean_cooking_pct


,coverage_band,observations,entities,first_year,last_year,female_hours_mean,male_hours_mean,time_gap_mean,time_gap_median,time_gap_q25,time_gap_q75,flagged_pairs
0,<50%,14,14,2003,2018,3.601,1.075,2.526,2.658,1.900,3.021,0
1,50–<80%,11,11,2001,2019,4.284,1.661,2.623,2.461,2.225,2.722,0
2,80–<95%,12,12,2007,2019,4.714,1.658,3.057,3.133,2.169,3.974,1
3,95–100%,49,49,2001,2019,4.294,1.933,2.361,2.317,1.507,2.880,2


Observations without infrastructure values: 3


In [10]:
BACKGROUND_VARIABLES = [
    "log_gdp_per_capita",
    "fertility_rate",
    "urban_population_pct",
]

FOCUS_VARIABLES = [
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "female_lfpr_pct",
    "lfpr_gap_pp",
    "electricity_pct",
    "water_pct",
    "clean_cooking_pct",
]

background_views = {}
background_rows = []

for view_name, original in association_views.items():
    frame = original.copy()
    gdp = frame["gdp_per_capita_ppp"]

    if (gdp.notna() & gdp.le(0)).any():
        raise ValueError("GDP must be positive before log transformation.")

    frame["log_gdp_per_capita"] = np.log(gdp)
    background_views[view_name] = frame

    for background in BACKGROUND_VARIABLES:
        for outcome in FOCUS_VARIABLES:
            pair = (
                frame[["iso3", background, outcome]]
                .replace([np.inf, -np.inf], np.nan)
                .dropna(subset=[background, outcome])
            )

            background_rows.append({
                "view": view_name,
                "background_variable": background,
                "related_variable": outcome,
                "observations": len(pair),
                "entities": pair["iso3"].nunique(),
                "pearson_r": calculate_correlation(
                    pair, background, outcome, "pearson"
                ),
                "spearman_rho": calculate_correlation(
                    pair, background, outcome, "spearman"
                ),
            })

background_associations = pd.DataFrame(background_rows)

for view_name in background_views:
    print(f"\nBackground associations: {view_name}")

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
    ):
        display(
            background_associations.loc[
                background_associations["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Background associations: All available observations


,background_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,log_gdp_per_capita,female_unpaid_hours,175,88,-0.105,-0.250
1,log_gdp_per_capita,male_unpaid_hours,175,88,0.425,0.477
2,log_gdp_per_capita,time_gap_hours,175,88,-0.425,-0.561
3,log_gdp_per_capita,female_lfpr_pct,175,88,-0.028,0.130
4,log_gdp_per_capita,lfpr_gap_pp,175,88,-0.130,-0.137
5,log_gdp_per_capita,electricity_pct,175,88,0.698,0.865
6,log_gdp_per_capita,water_pct,172,88,0.793,0.838
7,log_gdp_per_capita,clean_cooking_pct,169,85,0.829,0.873
8,fertility_rate,female_unpaid_hours,177,89,-0.029,0.063
9,fertility_rate,male_unpaid_hours,177,89,-0.349,-0.239



Background associations: Exclude source conflicts


,background_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,log_gdp_per_capita,female_unpaid_hours,170,88,-0.113,-0.265
1,log_gdp_per_capita,male_unpaid_hours,170,88,0.420,0.467
2,log_gdp_per_capita,time_gap_hours,170,88,-0.431,-0.573
3,log_gdp_per_capita,female_lfpr_pct,170,88,-0.030,0.136
4,log_gdp_per_capita,lfpr_gap_pp,170,88,-0.126,-0.126
5,log_gdp_per_capita,electricity_pct,170,88,0.704,0.866
6,log_gdp_per_capita,water_pct,167,88,0.795,0.838
7,log_gdp_per_capita,clean_cooking_pct,164,85,0.829,0.872
8,fertility_rate,female_unpaid_hours,172,89,-0.048,0.052
9,fertility_rate,male_unpaid_hours,172,89,-0.356,-0.250



Background associations: Latest observation per entity


,background_variable,related_variable,observations,entities,pearson_r,spearman_rho
0,log_gdp_per_capita,female_unpaid_hours,88,88,0.027,-0.152
1,log_gdp_per_capita,male_unpaid_hours,88,88,0.482,0.504
2,log_gdp_per_capita,time_gap_hours,88,88,-0.321,-0.432
3,log_gdp_per_capita,female_lfpr_pct,88,88,-0.113,0.009
4,log_gdp_per_capita,lfpr_gap_pp,88,88,-0.111,-0.160
5,log_gdp_per_capita,electricity_pct,88,88,0.758,0.873
6,log_gdp_per_capita,water_pct,88,88,0.812,0.846
7,log_gdp_per_capita,clean_cooking_pct,85,85,0.834,0.879
8,fertility_rate,female_unpaid_hours,89,89,-0.095,0.054
9,fertility_rate,male_unpaid_hours,89,89,-0.419,-0.350


In [11]:
gdp_groups = background_views[
    "Latest observation per entity"
].dropna(subset=["gdp_per_capita_ppp"]).copy()

GDP_GROUP_LABELS = [
    "Lower GDP third",
    "Middle GDP third",
    "Upper GDP third",
]

gdp_groups["gdp_group"], gdp_edges = pd.qcut(
    gdp_groups["gdp_per_capita_ppp"],
    q=3,
    labels=GDP_GROUP_LABELS,
    retbins=True,
)

print("GDP cut points: constant 2021 international dollars")
display(pd.DataFrame({
    "cut_point": ["Minimum", "One-third", "Two-thirds", "Maximum"],
    "gdp_per_capita_ppp": gdp_edges,
}).round(2))

gdp_group_summary = (
    gdp_groups.groupby("gdp_group", observed=True)
    .agg(
        entities=("iso3", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        gdp_median=("gdp_per_capita_ppp", "median"),
        female_hours_mean=("female_unpaid_hours", "mean"),
        male_hours_mean=("male_unpaid_hours", "mean"),
        time_gap_mean=("time_gap_hours", "mean"),
        time_gap_median=("time_gap_hours", "median"),
        female_lfpr_mean=("female_lfpr_pct", "mean"),
        lfpr_gap_mean=("lfpr_gap_pp", "mean"),
        fertility_median=("fertility_rate", "median"),
        urban_population_median=("urban_population_pct", "median"),
        flagged_pairs=("cross_source_conflict", "sum"),
    )
    .reset_index()
)

print("\nDescriptive statistics by relative GDP group")
with pd.option_context("display.max_columns", None):
    display(gdp_group_summary.round(3))

within_group_rows = []

for group_name, frame in gdp_groups.groupby(
    "gdp_group", observed=True
):
    for outcome in ["female_lfpr_pct", "lfpr_gap_pp"]:
        pair = frame.dropna(subset=["time_gap_hours", outcome])

        within_group_rows.append({
            "gdp_group": group_name,
            "related_variable": outcome,
            "observations": len(pair),
            "entities": pair["iso3"].nunique(),
            "pearson_r": calculate_correlation(
                pair, "time_gap_hours", outcome, "pearson"
            ),
            "spearman_rho": calculate_correlation(
                pair, "time_gap_hours", outcome, "spearman"
            ),
        })

print("\nTime Tax associations within relative GDP groups")
display(pd.DataFrame(within_group_rows).round(3))

print(
    "\nEntities excluded because GDP is missing:",
    len(background_views["Latest observation per entity"]) - len(gdp_groups),
)

GDP cut points: constant 2021 international dollars


,cut_point,gdp_per_capita_ppp
0,Minimum,1214.74
1,One-third,14551.88
2,Two-thirds,30862.90
3,Maximum,138046.60



Descriptive statistics by relative GDP group


,gdp_group,entities,first_year,last_year,gdp_median,female_hours_mean,male_hours_mean,time_gap_mean,time_gap_median,female_lfpr_mean,lfpr_gap_mean,fertility_median,urban_population_median,flagged_pairs
0,Lower GDP third,30,2001,2018,7559.898,4.288,1.253,3.035,2.992,50.899,25.108,3.186,50.393,0
1,Middle GDP third,29,2001,2019,20310.712,4.432,1.783,2.648,2.461,47.826,22.832,1.724,67.949,2
2,Upper GDP third,29,2002,2019,54017.946,3.938,2.047,1.891,1.800,53.344,16.383,1.720,80.473,1



Time Tax associations within relative GDP groups


,gdp_group,related_variable,observations,entities,pearson_r,spearman_rho
0,Lower GDP third,female_lfpr_pct,30,30,-0.663,-0.600
1,Lower GDP third,lfpr_gap_pp,30,30,0.719,0.746
2,Middle GDP third,female_lfpr_pct,29,29,-0.575,-0.484
3,Middle GDP third,lfpr_gap_pp,29,29,0.584,0.570
4,Upper GDP third,female_lfpr_pct,29,29,-0.528,-0.503
5,Upper GDP third,lfpr_gap_pp,29,29,0.470,0.734



Entities excluded because GDP is missing: 1


In [12]:
CONTROL_SETS = {
    "GDP + year": [
        "log_gdp_per_capita",
        "year",
    ],
    "GDP + year + fertility + urbanization": [
        "log_gdp_per_capita",
        "year",
        "fertility_rate",
        "urban_population_pct",
    ],
}

PARTIAL_TARGETS = [
    "female_lfpr_pct",
    "lfpr_gap_pp",
    "electricity_pct",
    "water_pct",
    "clean_cooking_pct",
]

ALL_CONTROLS = list(dict.fromkeys(
    variable
    for controls in CONTROL_SETS.values()
    for variable in controls
))


def partial_pearson(frame, x_column, y_column, controls):
    control_values = frame[controls].to_numpy(dtype=float)
    control_sd = control_values.std(axis=0)

    if np.any(control_sd == 0):
        raise ValueError("A control variable has no variation.")

    # Standardization improves numerical stability.
    standardized = (
        control_values - control_values.mean(axis=0)
    ) / control_sd

    design = np.column_stack([
        np.ones(len(frame)),
        standardized,
    ])

    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError("Control matrix is not full rank.")

    if len(frame) <= design.shape[1] + 1:
        raise ValueError("Too few observations for these controls.")

    x = frame[x_column].to_numpy(dtype=float)
    y = frame[y_column].to_numpy(dtype=float)

    x_residual = x - design @ np.linalg.lstsq(
        design, x, rcond=None
    )[0]
    y_residual = y - design @ np.linalg.lstsq(
        design, y, rcond=None
    )[0]

    for values, residual in [(x, x_residual), (y, y_residual)]:
        original_norm = np.linalg.norm(values - values.mean())
        if (
            original_norm == 0
            or np.linalg.norm(residual) <= 1e-10 * original_norm
        ):
            return np.nan

    return float(np.corrcoef(x_residual, y_residual)[0, 1])


partial_rows = []
partial_analysis_data = {}

for view_name, frame in background_views.items():
    for target in PARTIAL_TARGETS:
        required = ["time_gap_hours", target] + ALL_CONTROLS

        numeric = (
            frame[required]
            .apply(pd.to_numeric, errors="raise")
            .replace([np.inf, -np.inf], np.nan)
        )

        complete = numeric.notna().all(axis=1)

        analysis = frame.loc[
            complete, ["iso3", "country"]
        ].join(numeric.loc[complete])

        partial_analysis_data[(view_name, target)] = analysis

        raw_r = calculate_correlation(
            analysis, "time_gap_hours", target, "pearson"
        )

        for control_name, controls in CONTROL_SETS.items():
            adjusted_r = partial_pearson(
                analysis,
                "time_gap_hours",
                target,
                controls,
            )

            partial_rows.append({
                "view": view_name,
                "related_variable": target,
                "controls": control_name,
                "observations": len(analysis),
                "entities": analysis["iso3"].nunique(),
                "excluded_incomplete_rows": len(frame) - len(analysis),
                "raw_pearson_r": raw_r,
                "partial_pearson_r": adjusted_r,
                "change_in_absolute_r": abs(adjusted_r) - abs(raw_r),
            })

partial_results = pd.DataFrame(partial_rows)

for view_name in background_views:
    print(f"\nPartial correlations: {view_name}")

    with pd.option_context(
        "display.max_columns", None,
        "display.max_rows", None,
    ):
        display(
            partial_results.loc[
                partial_results["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Partial correlations: All available observations


,related_variable,controls,observations,entities,excluded_incomplete_rows,raw_pearson_r,partial_pearson_r,change_in_absolute_r
0,female_lfpr_pct,GDP + year,175,88,2,-0.561,-0.632,0.071
1,female_lfpr_pct,GDP + year + fertility + urbanization,175,88,2,-0.561,-0.600,0.039
2,lfpr_gap_pp,GDP + year,175,88,2,0.672,0.688,0.016
3,lfpr_gap_pp,GDP + year + fertility + urbanization,175,88,2,0.672,0.690,0.018
4,electricity_pct,GDP + year,175,88,2,0.017,0.489,0.472
5,electricity_pct,GDP + year + fertility + urbanization,175,88,2,0.017,0.475,0.458
6,water_pct,GDP + year,172,88,5,-0.149,0.333,0.185
7,water_pct,GDP + year + fertility + urbanization,172,88,5,-0.149,0.273,0.124
8,clean_cooking_pct,GDP + year,169,85,8,-0.105,0.443,0.338
9,clean_cooking_pct,GDP + year + fertility + urbanization,169,85,8,-0.105,0.379,0.274



Partial correlations: Exclude source conflicts


,related_variable,controls,observations,entities,excluded_incomplete_rows,raw_pearson_r,partial_pearson_r,change_in_absolute_r
0,female_lfpr_pct,GDP + year,170,88,2,-0.563,-0.638,0.075
1,female_lfpr_pct,GDP + year + fertility + urbanization,170,88,2,-0.563,-0.601,0.039
2,lfpr_gap_pp,GDP + year,170,88,2,0.684,0.703,0.019
3,lfpr_gap_pp,GDP + year + fertility + urbanization,170,88,2,0.684,0.710,0.026
4,electricity_pct,GDP + year,170,88,2,0.028,0.519,0.491
5,electricity_pct,GDP + year + fertility + urbanization,170,88,2,0.028,0.493,0.465
6,water_pct,GDP + year,167,88,5,-0.140,0.360,0.220
7,water_pct,GDP + year + fertility + urbanization,167,88,5,-0.140,0.283,0.144
8,clean_cooking_pct,GDP + year,164,85,8,-0.095,0.480,0.385
9,clean_cooking_pct,GDP + year + fertility + urbanization,164,85,8,-0.095,0.404,0.309



Partial correlations: Latest observation per entity


,related_variable,controls,observations,entities,excluded_incomplete_rows,raw_pearson_r,partial_pearson_r,change_in_absolute_r
0,female_lfpr_pct,GDP + year,88,88,1,-0.578,-0.649,0.071
1,female_lfpr_pct,GDP + year + fertility + urbanization,88,88,1,-0.578,-0.632,0.054
2,lfpr_gap_pp,GDP + year,88,88,1,0.663,0.664,0.002
3,lfpr_gap_pp,GDP + year + fertility + urbanization,88,88,1,0.663,0.668,0.005
4,electricity_pct,GDP + year,88,88,1,0.066,0.525,0.459
5,electricity_pct,GDP + year + fertility + urbanization,88,88,1,0.066,0.534,0.468
6,water_pct,GDP + year,88,88,1,-0.109,0.274,0.165
7,water_pct,GDP + year + fertility + urbanization,88,88,1,-0.109,0.235,0.126
8,clean_cooking_pct,GDP + year,85,85,4,-0.007,0.454,0.447
9,clean_cooking_pct,GDP + year + fertility + urbanization,85,85,4,-0.007,0.445,0.437


In [13]:
comparison_index = [
    "view",
    "related_variable",
    "observations",
    "entities",
    "raw_pearson_r",
]

partial_comparison = (
    partial_results.pivot(
        index=comparison_index,
        columns="controls",
        values="partial_pearson_r",
    )
    .reset_index()
    .rename(columns={
        "raw_pearson_r": "raw_r",
        "GDP + year": "partial_gdp_year",
        "GDP + year + fertility + urbanization":
            "partial_full_background",
    })
)

partial_comparison.columns.name = None

for view_name in background_views:
    print(f"\n{view_name}")

    display(
        partial_comparison.loc[
            partial_comparison["view"].eq(view_name)
        ]
        .drop(columns="view")
        .reset_index(drop=True)
        .round(3)
    )


All available observations


,related_variable,observations,entities,raw_r,partial_gdp_year,partial_full_background
0,clean_cooking_pct,169,85,-0.105,0.443,0.379
1,electricity_pct,175,88,0.017,0.489,0.475
2,female_lfpr_pct,175,88,-0.561,-0.632,-0.600
3,lfpr_gap_pp,175,88,0.672,0.688,0.690
4,water_pct,172,88,-0.149,0.333,0.273



Exclude source conflicts


,related_variable,observations,entities,raw_r,partial_gdp_year,partial_full_background
0,clean_cooking_pct,164,85,-0.095,0.480,0.404
1,electricity_pct,170,88,0.028,0.519,0.493
2,female_lfpr_pct,170,88,-0.563,-0.638,-0.601
3,lfpr_gap_pp,170,88,0.684,0.703,0.710
4,water_pct,167,88,-0.140,0.360,0.283



Latest observation per entity


,related_variable,observations,entities,raw_r,partial_gdp_year,partial_full_background
0,clean_cooking_pct,85,85,-0.007,0.454,0.445
1,electricity_pct,88,88,0.066,0.525,0.534
2,female_lfpr_pct,88,88,-0.578,-0.649,-0.632
3,lfpr_gap_pp,88,88,0.663,0.664,0.668
4,water_pct,88,88,-0.109,0.274,0.235


In [14]:
ADJUSTMENT_STEPS = {
    "Unadjusted": [],
    "Year only": ["year"],
    "GDP only": ["log_gdp_per_capita"],
    "GDP + year": ["log_gdp_per_capita", "year"],
    "Full background": [
        "log_gdp_per_capita",
        "year",
        "fertility_rate",
        "urban_population_pct",
    ],
}

TIME_COMPONENTS = [
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
]

FULL_CONTROLS = ADJUSTMENT_STEPS["Full background"]

reversal_rows = []
reversal_data = {}

for view_name, frame in background_views.items():
    for infrastructure in INFRASTRUCTURE_VARIABLES:
        required = TIME_COMPONENTS + [infrastructure] + FULL_CONTROLS

        numeric = (
            frame[required]
            .apply(pd.to_numeric, errors="raise")
            .replace([np.inf, -np.inf], np.nan)
        )

        complete = numeric.notna().all(axis=1)

        analysis = frame.loc[
            complete, ["iso3", "country"]
        ].join(numeric.loc[complete])

        reversal_data[(view_name, infrastructure)] = analysis

        for time_variable in TIME_COMPONENTS:
            for step_name, controls in ADJUSTMENT_STEPS.items():
                if controls:
                    coefficient = partial_pearson(
                        analysis,
                        time_variable,
                        infrastructure,
                        controls,
                    )
                else:
                    coefficient = calculate_correlation(
                        analysis,
                        time_variable,
                        infrastructure,
                        "pearson",
                    )

                reversal_rows.append({
                    "view": view_name,
                    "infrastructure": infrastructure,
                    "time_variable": time_variable,
                    "adjustment": step_name,
                    "observations": len(analysis),
                    "entities": analysis["iso3"].nunique(),
                    "correlation": coefficient,
                })

reversal_results = pd.DataFrame(reversal_rows)

for view_name in background_views:
    table = (
        reversal_results.loc[
            reversal_results["view"].eq(view_name)
        ]
        .pivot(
            index=[
                "infrastructure",
                "time_variable",
                "observations",
                "entities",
            ],
            columns="adjustment",
            values="correlation",
        )
        .reindex(columns=list(ADJUSTMENT_STEPS))
        .reset_index()
    )

    print(f"\nAdjustment sequence: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(table.round(3))


Adjustment sequence: All available observations


adjustment,infrastructure,time_variable,observations,entities,Unadjusted,Year only,GDP only,GDP + year,Full background
0,clean_cooking_pct,female_unpaid_hours,169,85,0.181,0.175,0.437,0.423,0.373
1,clean_cooking_pct,male_unpaid_hours,169,85,0.378,0.376,0.038,0.010,0.036
2,clean_cooking_pct,time_gap_hours,169,85,-0.105,-0.108,0.445,0.443,0.379
3,electricity_pct,female_unpaid_hours,175,88,0.263,0.284,0.473,0.495,0.480
4,electricity_pct,male_unpaid_hours,175,88,0.323,0.341,0.039,0.053,0.069
5,electricity_pct,time_gap_hours,175,88,0.017,0.022,0.484,0.489,0.475
6,water_pct,female_unpaid_hours,172,88,0.074,0.083,0.255,0.259,0.203
7,water_pct,male_unpaid_hours,172,88,0.294,0.304,-0.073,-0.077,-0.066
8,water_pct,time_gap_hours,172,88,-0.149,-0.146,0.333,0.333,0.273



Adjustment sequence: Exclude source conflicts


adjustment,infrastructure,time_variable,observations,entities,Unadjusted,Year only,GDP only,GDP + year,Full background
0,clean_cooking_pct,female_unpaid_hours,164,85,0.201,0.194,0.484,0.471,0.403
1,clean_cooking_pct,male_unpaid_hours,164,85,0.382,0.378,0.054,0.024,0.040
2,clean_cooking_pct,time_gap_hours,164,85,-0.095,-0.097,0.477,0.480,0.404
3,electricity_pct,female_unpaid_hours,170,88,0.287,0.303,0.520,0.536,0.504
4,electricity_pct,male_unpaid_hours,170,88,0.329,0.344,0.051,0.062,0.072
5,electricity_pct,time_gap_hours,170,88,0.028,0.031,0.517,0.519,0.493
6,water_pct,female_unpaid_hours,167,88,0.089,0.095,0.292,0.292,0.213
7,water_pct,male_unpaid_hours,167,88,0.297,0.305,-0.062,-0.067,-0.065
8,water_pct,time_gap_hours,167,88,-0.140,-0.138,0.360,0.360,0.283



Adjustment sequence: Latest observation per entity


adjustment,infrastructure,time_variable,observations,entities,Unadjusted,Year only,GDP only,GDP + year,Full background
0,clean_cooking_pct,female_unpaid_hours,85,85,0.296,0.303,0.447,0.422,0.403
1,clean_cooking_pct,male_unpaid_hours,85,85,0.395,0.396,-0.044,-0.078,-0.088
2,clean_cooking_pct,time_gap_hours,85,85,-0.007,-0.006,0.462,0.454,0.445
3,electricity_pct,female_unpaid_hours,88,88,0.325,0.384,0.467,0.517,0.516
4,electricity_pct,male_unpaid_hours,88,88,0.330,0.351,-0.063,-0.038,-0.044
5,electricity_pct,time_gap_hours,88,88,0.066,0.100,0.500,0.525,0.534
6,water_pct,female_unpaid_hours,88,88,0.131,0.156,0.187,0.186,0.139
7,water_pct,male_unpaid_hours,88,88,0.322,0.330,-0.137,-0.142,-0.150
8,water_pct,time_gap_hours,88,88,-0.109,-0.095,0.275,0.274,0.235


In [15]:
partial_loo_rows = []
partial_loo_summary_rows = []

for (view_name, infrastructure), analysis in reversal_data.items():
    baseline = partial_pearson(
        analysis,
        "time_gap_hours",
        infrastructure,
        FULL_CONTROLS,
    )

    current_rows = []

    for iso3 in sorted(analysis["iso3"].unique()):
        remaining = analysis.loc[analysis["iso3"].ne(iso3)]

        country = analysis.loc[
            analysis["iso3"].eq(iso3), "country"
        ].iloc[0]

        try:
            coefficient = partial_pearson(
                remaining,
                "time_gap_hours",
                infrastructure,
                FULL_CONTROLS,
            )
            diagnostic = (
                "" if np.isfinite(coefficient)
                else "Undefined residual correlation"
            )
        except ValueError as error:
            coefficient = np.nan
            diagnostic = str(error)

        result = {
            "view": view_name,
            "infrastructure": infrastructure,
            "excluded_iso3": iso3,
            "excluded_country": country,
            "removed_observations": len(analysis) - len(remaining),
            "baseline": baseline,
            "partial_r_after_exclusion": coefficient,
            "absolute_change": abs(coefficient - baseline),
            "diagnostic": diagnostic,
        }

        current_rows.append(result)
        partial_loo_rows.append(result)

    current = pd.DataFrame(current_rows)
    valid = current.dropna(subset=["partial_r_after_exclusion"])

    if valid.empty:
        raise ValueError(
            f"No valid exclusions: {view_name}, {infrastructure}"
        )

    largest_change = valid.loc[valid["absolute_change"].idxmax()]

    partial_loo_summary_rows.append({
        "view": view_name,
        "infrastructure": infrastructure,
        "baseline_partial_r": baseline,
        "minimum_after_exclusion":
            valid["partial_r_after_exclusion"].min(),
        "maximum_after_exclusion":
            valid["partial_r_after_exclusion"].max(),
        "largest_absolute_change":
            largest_change["absolute_change"],
        "most_influential_exclusion":
            largest_change["excluded_country"],
        "undefined_exclusions": len(current) - len(valid),
    })

partial_loo_details = pd.DataFrame(partial_loo_rows)
partial_loo_summary = pd.DataFrame(partial_loo_summary_rows)

for view_name in background_views:
    print(f"\nPartial-correlation sensitivity: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            partial_loo_summary.loc[
                partial_loo_summary["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )

diagnostics = partial_loo_details.loc[
    partial_loo_details["diagnostic"].ne("")
]

if not diagnostics.empty:
    print("\nExclusions requiring review")
    display(diagnostics)


Partial-correlation sensitivity: All available observations


,infrastructure,baseline_partial_r,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion,undefined_exclusions
0,electricity_pct,0.475,0.425,0.523,0.050,Liberia,0
1,water_pct,0.273,0.232,0.300,0.040,"Egypt, Arab Rep.",0
2,clean_cooking_pct,0.379,0.330,0.413,0.049,Lao PDR,0



Partial-correlation sensitivity: Exclude source conflicts


,infrastructure,baseline_partial_r,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion,undefined_exclusions
0,electricity_pct,0.493,0.441,0.546,0.053,Kyrgyz Republic,0
1,water_pct,0.283,0.241,0.314,0.042,"Egypt, Arab Rep.",0
2,clean_cooking_pct,0.404,0.354,0.440,0.050,Lao PDR,0



Partial-correlation sensitivity: Latest observation per entity


,infrastructure,baseline_partial_r,minimum_after_exclusion,maximum_after_exclusion,largest_absolute_change,most_influential_exclusion,undefined_exclusions
0,electricity_pct,0.534,0.475,0.579,0.059,Liberia,0
1,water_pct,0.235,0.157,0.268,0.077,"Egypt, Arab Rep.",0
2,clean_cooking_pct,0.445,0.373,0.491,0.071,"Egypt, Arab Rep.",0


In [16]:
from pathlib import Path
import pandas as pd

RAW_ROOT = PREPARED_DIR.parent / "all_sources"
GENDER_FOLDER = RAW_ROOT / "014_GDS"

gender_catalog = pd.read_csv(
    GENDER_FOLDER / "_indicators.csv",
    dtype="string",
    keep_default_na=False,
)

JOB_QUALITY_SPECS = [
    ("female_vulnerable_pct", "SL.EMP.VULN.FE.ZS"),
    ("male_vulnerable_pct", "SL.EMP.VULN.MA.ZS"),
    ("female_wage_salaried_pct", "SL.EMP.WORK.FE.ZS"),
    ("male_wage_salaried_pct", "SL.EMP.WORK.MA.ZS"),
    ("female_self_employed_pct", "SL.EMP.SELF.FE.ZS"),
    ("male_self_employed_pct", "SL.EMP.SELF.MA.ZS"),
]

metadata_rows = []

for variable, code in JOB_QUALITY_SPECS:
    matches = gender_catalog.loc[gender_catalog["id"].eq(code)]

    result = {
        "variable": variable,
        "indicator": code,
        "catalog_matches": len(matches),
        "file_exists": (GENDER_FOLDER / f"{code}.csv.gz").is_file(),
    }

    for field in ["name", "unit", "definition", "source_org"]:
        result[field] = (
            matches.iloc[0].get(field, "")
            if len(matches) == 1 else ""
        )

    metadata_rows.append(result)

job_quality_dictionary = pd.DataFrame(metadata_rows)

display(
    job_quality_dictionary[
        ["variable", "indicator", "catalog_matches", "file_exists", "name"]
    ]
)

for row in job_quality_dictionary.itertuples(index=False):
    print(f"\nIndicator: {row.indicator}")
    print(f"Name: {row.name or '[Missing]'}")
    print(f"Unit field: {row.unit or '[Not provided]'}")
    print(f"Definition: {row.definition or '[Not provided]'}")
    print(f"Source: {row.source_org or '[Not provided]'}")

,variable,indicator,catalog_matches,file_exists,name
0,female_vulnerable_pct,SL.EMP.VULN.FE.ZS,1,True,"Vulnerable employment, female (% of female emp..."
1,male_vulnerable_pct,SL.EMP.VULN.MA.ZS,1,True,"Vulnerable employment, male (% of male employm..."
2,female_wage_salaried_pct,SL.EMP.WORK.FE.ZS,1,True,"Wage and salaried workers, female (% of female..."
3,male_wage_salaried_pct,SL.EMP.WORK.MA.ZS,1,True,"Wage and salaried workers, male (% of male emp..."
4,female_self_employed_pct,SL.EMP.SELF.FE.ZS,1,True,"Self-employed, female (% of female employment)..."
5,male_self_employed_pct,SL.EMP.SELF.MA.ZS,1,True,"Self-employed, male (% of male employment) (mo..."



Indicator: SL.EMP.VULN.FE.ZS
Name: Vulnerable employment, female (% of female employment) (modeled ILO estimate)
Unit field: [Not provided]
Definition: Vulnerable employment is contributing family workers and own-account workers as a percentage of total employment.
Source: ILO Modelled Estimates database (ILOEST), International Labour Organization (ILO), uri: https://ilostat.ilo.org/data/bulk/, publisher: ILOSTAT, type: estimates based on external database;
Staff estimates, World Bank (WB)

Indicator: SL.EMP.VULN.MA.ZS
Name: Vulnerable employment, male (% of male employment) (modeled ILO estimate)
Unit field: [Not provided]
Definition: Vulnerable employment is contributing family workers and own-account workers as a percentage of total employment.
Source: ILO Modelled Estimates database (ILOEST), International Labour Organization (ILO), uri: https://ilostat.ilo.org/data/bulk/, publisher: ILOSTAT, type: estimates based on external database;
Staff estimates, World Bank (WB)

Indicator: 

In [17]:
mapping_raw = pd.read_csv(
    RAW_ROOT / "002_WDI" / "EG.ELC.ACCS.ZS.csv.gz",
    dtype="string",
    keep_default_na=False,
)

crosswalk = (
    mapping_raw[["country_id", "iso3"]]
    .apply(lambda column: column.str.strip())
    .query("country_id != '' and iso3 != ''")
    .drop_duplicates()
)

if (
    crosswalk["country_id"].duplicated().any()
    or crosswalk["iso3"].duplicated().any()
):
    raise ValueError("Country-code mapping is not one-to-one.")

id_to_iso3 = crosswalk.set_index("country_id")["iso3"].to_dict()
known_iso3 = set(crosswalk["iso3"])
target_entities = set(eda["iso3"])

if target_entities - known_iso3:
    raise ValueError("Some time-use entities are absent from the crosswalk.")

employment_eda = eda.copy()
job_coverage_rows = []

for row in job_quality_dictionary.itertuples(index=False):
    if row.catalog_matches != 1 or not row.file_exists:
        print(f"Unavailable indicator: {row.indicator}")
        continue

    raw = pd.read_csv(
        GENDER_FOLDER / f"{row.indicator}.csv.gz",
        dtype="string",
        keep_default_na=False,
    )

    country_id = raw["country_id"].str.strip()
    reported_iso3 = raw["iso3"].str.strip()

    mapped_iso3 = country_id.map(id_to_iso3)
    mapped_iso3 = mapped_iso3.mask(
        country_id.isin(known_iso3), country_id
    )

    conflict = (
        reported_iso3.ne("")
        & mapped_iso3.notna()
        & reported_iso3.ne(mapped_iso3).fillna(False)
    )
    if conflict.any():
        raise ValueError(f"Conflicting country codes: {row.indicator}")

    resolved_iso3 = reported_iso3.mask(
        reported_iso3.eq(""), mapped_iso3
    )
    keep = resolved_iso3.isin(target_entities)

    selected = raw.loc[keep].copy()
    year_text = selected["date"].str.strip()

    if not year_text.str.fullmatch(r"\d{4}", na=False).all():
        raise ValueError(f"Nonannual dates: {row.indicator}")

    prepared = pd.DataFrame({
        "iso3": resolved_iso3.loc[keep],
        "year": year_text.astype(int),
        row.variable: pd.to_numeric(
            selected["value"].str.strip().replace("", pd.NA),
            errors="raise",
        ),
    })

    if prepared.duplicated(["iso3", "year"]).any():
        raise ValueError(f"Duplicate country-year keys: {row.indicator}")

    employment_eda = employment_eda.merge(
        prepared,
        on=["iso3", "year"],
        how="left",
        validate="one_to_one",
    )

    values = employment_eda[row.variable]
    available = values.notna()

    job_coverage_rows.append({
        "variable": row.variable,
        "matched_observations": int(available.sum()),
        "matched_entities": employment_eda.loc[
            available, "iso3"
        ].nunique(),
        "missing_observations": int((~available).sum()),
        "minimum": values.min(),
        "maximum": values.max(),
        "outside_0_to_100": int(
            (available & ~values.between(0, 100).fillna(False)).sum()
        ),
    })

job_quality_coverage = pd.DataFrame(job_coverage_rows)

display(job_quality_coverage.round(3))

if len(employment_eda) != len(eda):
    raise ValueError("The number of anchor observations changed.")

print(f"Time-use observations retained: {len(employment_eda)}")

,variable,matched_observations,matched_entities,missing_observations,minimum,maximum,outside_0_to_100
0,female_vulnerable_pct,177,89,0,0.267,93.256,0
1,male_vulnerable_pct,177,89,0,0.237,85.179,0
2,female_wage_salaried_pct,177,89,0,5.929,99.298,0
3,male_wage_salaried_pct,177,89,0,14.029,99.483,0
4,female_self_employed_pct,177,89,0,0.701,94.071,0
5,male_self_employed_pct,177,89,0,0.517,85.971,0


Time-use observations retained: 177


In [18]:
employment_analysis = employment_eda.copy()

complement_rows = []

for sex in ["female", "male"]:
    wage = employment_analysis[f"{sex}_wage_salaried_pct"]
    self_employed = employment_analysis[f"{sex}_self_employed_pct"]
    vulnerable = employment_analysis[f"{sex}_vulnerable_pct"]

    deviation = (wage + self_employed - 100).abs()

    complement_rows.append({
        "sex": sex,
        "observations": len(employment_analysis),
        "maximum_deviation_from_100": deviation.max(),
        "rows_with_deviation_over_0_01": int(
            deviation.gt(0.01).sum()
        ),
        "rows_vulnerable_above_self_employed": int(
            (vulnerable - self_employed).gt(0.01).sum()
        ),
    })

display(pd.DataFrame(complement_rows).round(5))

for employment_type in ["vulnerable", "wage_salaried", "self_employed"]:
    employment_analysis[f"{employment_type}_gap_pp"] = (
        employment_analysis[f"female_{employment_type}_pct"]
        - employment_analysis[f"male_{employment_type}_pct"]
    )

gdp = employment_analysis["gdp_per_capita_ppp"]

if (gdp.notna() & gdp.le(0)).any():
    raise ValueError("GDP must be positive before log transformation.")

employment_analysis["log_gdp_per_capita"] = np.log(gdp)

employment_views = {
    "All available observations": employment_analysis,
    "Exclude source conflicts": employment_analysis.loc[
        ~employment_analysis["cross_source_conflict"]
    ].copy(),
    "Latest observation per entity": (
        employment_analysis
        .sort_values(["iso3", "year"])
        .drop_duplicates("iso3", keep="last")
        .copy()
    ),
}

display(
    employment_analysis[
        [
            "country",
            "year",
            "time_gap_hours",
            "vulnerable_gap_pp",
            "wage_salaried_gap_pp",
            "self_employed_gap_pp",
        ]
    ].head().round(3)
)

,sex,observations,maximum_deviation_from_100,rows_with_deviation_over_0_01,rows_vulnerable_above_self_employed
0,female,177,0.00067,0,0
1,male,177,0.00027,0,0


,country,year,time_gap_hours,vulnerable_gap_pp,wage_salaried_gap_pp,self_employed_gap_pp
0,Albania,2011,4.383,8.379,-7.1,7.1
1,Argentina,2005,2.550,-5.011,7.891,-7.891
2,Argentina,2010,2.617,-4.357,7.043,-7.043
3,Argentina,2013,3.401,-5.067,7.437,-7.437
4,Armenia,2004,4.667,9.446,-8.332,8.332


In [19]:
EMPLOYMENT_TARGETS = [
    "female_vulnerable_pct",
    "female_wage_salaried_pct",
    "female_self_employed_pct",
    "vulnerable_gap_pp",
    "wage_salaried_gap_pp",
    "self_employed_gap_pp",
]

EMPLOYMENT_CONTROLS = [
    "log_gdp_per_capita",
    "year",
    "fertility_rate",
    "urban_population_pct",
]

employment_correlation_rows = []

for view_name, frame in employment_views.items():
    for target in EMPLOYMENT_TARGETS:
        required = [
            "time_gap_hours",
            target,
        ] + EMPLOYMENT_CONTROLS

        numeric = (
            frame[required]
            .apply(pd.to_numeric, errors="raise")
            .replace([np.inf, -np.inf], np.nan)
        )

        complete = numeric.notna().all(axis=1)

        analysis = frame.loc[
            complete, ["iso3", "country"]
        ].join(numeric.loc[complete])

        employment_correlation_rows.append({
            "view": view_name,
            "related_variable": target,
            "observations": len(analysis),
            "entities": analysis["iso3"].nunique(),
            "excluded_incomplete_rows": len(frame) - len(analysis),
            "pearson_r": calculate_correlation(
                analysis, "time_gap_hours", target, "pearson"
            ),
            "spearman_rho": calculate_correlation(
                analysis, "time_gap_hours", target, "spearman"
            ),
            "partial_pearson_r": partial_pearson(
                analysis,
                "time_gap_hours",
                target,
                EMPLOYMENT_CONTROLS,
            ),
        })

employment_correlations = pd.DataFrame(employment_correlation_rows)

for view_name in employment_views:
    print(f"\nTime Tax and employment structure: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            employment_correlations.loc[
                employment_correlations["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Time Tax and employment structure: All available observations


,related_variable,observations,entities,excluded_incomplete_rows,pearson_r,spearman_rho,partial_pearson_r
0,female_vulnerable_pct,175,88,2,0.335,0.570,0.044
1,female_wage_salaried_pct,175,88,2,-0.345,-0.568,-0.061
2,female_self_employed_pct,175,88,2,0.345,0.568,0.061
3,vulnerable_gap_pp,175,88,2,0.344,0.359,0.178
4,wage_salaried_gap_pp,175,88,2,-0.318,-0.352,-0.142
5,self_employed_gap_pp,175,88,2,0.318,0.352,0.142



Time Tax and employment structure: Exclude source conflicts


,related_variable,observations,entities,excluded_incomplete_rows,pearson_r,spearman_rho,partial_pearson_r
0,female_vulnerable_pct,170,88,2,0.341,0.585,0.054
1,female_wage_salaried_pct,170,88,2,-0.352,-0.584,-0.072
2,female_self_employed_pct,170,88,2,0.352,0.584,0.072
3,vulnerable_gap_pp,170,88,2,0.334,0.348,0.156
4,wage_salaried_gap_pp,170,88,2,-0.308,-0.341,-0.119
5,self_employed_gap_pp,170,88,2,0.308,0.341,0.119



Time Tax and employment structure: Latest observation per entity


,related_variable,observations,entities,excluded_incomplete_rows,pearson_r,spearman_rho,partial_pearson_r
0,female_vulnerable_pct,88,88,1,0.265,0.430,0.060
1,female_wage_salaried_pct,88,88,1,-0.273,-0.429,-0.072
2,female_self_employed_pct,88,88,1,0.273,0.429,0.072
3,vulnerable_gap_pp,88,88,1,0.292,0.294,0.155
4,wage_salaried_gap_pp,88,88,1,-0.275,-0.286,-0.136
5,self_employed_gap_pp,88,88,1,0.275,0.286,0.136


In [20]:
from pathlib import Path
import json
import pandas as pd

RESULTS_DIR = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\analysis_results\time_tax_eda"
)

TABLE_NAMES = [
    "quality_report",
    "associations",
    "loo_details",
    "loo_summary",
    "background_associations",
    "gdp_group_summary",
    "partial_results",
    "partial_comparison",
    "reversal_results",
    "partial_loo_details",
    "partial_loo_summary",
    "job_quality_dictionary",
    "job_quality_coverage",
    "employment_correlations",
]

DATA_NAMES = [
    "eda",
    "latest_per_entity",
    "employment_analysis",
]

DICTIONARY_NAMES = [
    "descriptive_tables",
    "infrastructure_group_tables",
]

required_names = TABLE_NAMES + DATA_NAMES + DICTIONARY_NAMES
missing_names = [
    name for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run the preceding notebook cells before exporting. "
        f"Missing variables: {missing_names}"
    )

# Prepare combined tables before writing any files.
descriptive_export = pd.concat(
    [
        table.rename_axis("variable")
        .reset_index()
        .assign(view=view_name)
        for view_name, table in descriptive_tables.items()
    ],
    ignore_index=True,
)

infrastructure_export = pd.concat(
    [
        table.assign(view=view_name, infrastructure=variable)
        for (view_name, variable), table
        in infrastructure_group_tables.items()
    ],
    ignore_index=True,
)

# Preserve the within-GDP-group correlations previously displayed.
within_gdp_export = pd.DataFrame(within_group_rows)

tables = {
    name: globals()[name]
    for name in TABLE_NAMES
}

tables.update({
    "descriptive_statistics": descriptive_export,
    "infrastructure_group_statistics": infrastructure_export,
    "within_gdp_group_correlations": within_gdp_export,
})

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
manifest_rows = []

for name, table in tables.items():
    path = RESULTS_DIR / f"{name}.csv"
    table.to_csv(path, index=False, encoding="utf-8-sig")

    manifest_rows.append({
        "file": path.name,
        "rows": len(table),
        "columns": len(table.columns),
        "type": "result_table",
    })

for name in DATA_NAMES:
    frame = globals()[name]
    path = RESULTS_DIR / f"{name}.csv.gz"

    frame.to_csv(path, index=False, compression="gzip")

    manifest_rows.append({
        "file": path.name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "type": "analysis_data",
    })

metadata = {
    "exported_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "time_source": "World Bank source 46: Sustainable Development Goals",
    "time_tax_definition": (
        "Female unpaid domestic and care hours per day "
        "minus male unpaid domestic and care hours per day"
    ),
    "lfpr_gap_definition": (
        "Male labor force participation percentage "
        "minus female labor force participation percentage"
    ),
    "employment_structure_gap_definition": (
        "Female employment-type percentage "
        "minus male employment-type percentage"
    ),
    "partial_correlation_controls": [
        "log_gdp_per_capita",
        "year",
        "fertility_rate",
        "urban_population_pct",
    ],
    "anchor_observations": len(eda),
    "anchor_entities": int(eda["iso3"].nunique()),
    "first_year": int(eda["year"].min()),
    "last_year": int(eda["year"].max()),
    "source_conflict_pairs": int(eda["cross_source_conflict"].sum()),
}

with open(
    RESULTS_DIR / "analysis_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

manifest = pd.DataFrame(manifest_rows)

# Verify that every listed output was written and is nonempty.
manifest["bytes"] = manifest["file"].map(
    lambda filename: (RESULTS_DIR / filename).stat().st_size
)

if manifest["bytes"].le(0).any():
    raise RuntimeError("An exported file is empty.")

manifest.to_csv(
    RESULTS_DIR / "export_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

display(manifest)
print(f"Saved {len(manifest)} data and result files.")
print("Also saved analysis_metadata.json and export_manifest.csv.")
print(f"Output directory: {RESULTS_DIR}")

,file,rows,columns,type,bytes
0,quality_report.csv,12,6,result_table,585
1,associations.csv,45,7,result_table,4896
2,loo_details.csv,2652,9,result_table,339820
3,loo_summary.csv,30,8,result_table,4398
4,background_associations.csv,72,7,result_table,7915
5,gdp_group_summary.csv,3,14,result_table,705
6,partial_results.csv,30,9,result_table,4198
7,partial_comparison.csv,15,7,result_table,1722
8,reversal_results.csv,135,7,result_table,13383
9,partial_loo_details.csv,783,9,result_table,93947


Saved 20 data and result files.
Also saved analysis_metadata.json and export_manifest.csv.
Output directory: D:\胡溪筼\Documents\2026-7Fall\Carolina Data Challenge 2026\Graduate_Dataset\analysis_results\time_tax_eda
